# Task 1: Biểu diễn ảnh màu và ảnh xám

Nội dung:
1. Load ảnh màu
2. Chuyển ảnh màu sang ảnh xám
3. Chuyển ảnh xám sang ảnh màu
4. Tách các kênh R, G, B
5. Kết hợp các kênh màu (tái tạo, hoán đổi, thiếu kênh, thay thế kênh)
6. Mở rộng: biến đổi màu bằng ma trận 3x3, không gian YCrCb
7. Nhận xét chung

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

IN_DIR = Path('input')
OUT_DIR = Path('output')
OUT_DIR.mkdir(exist_ok=True)


def load_rgb(path):
    # cv2 đọc ảnh theo thứ tự BGR, cần đổi sang RGB.
    # Ảnh có kênh alpha (RGBA) được ghép lên nền trắng.
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    if img.shape[2] == 4:
        alpha = img[:, :, 3:] / 255.0
        img = (img[:, :, :3] * alpha + 255 * (1 - alpha)).astype(np.uint8)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def show(images, titles, cols=4, save=None, size=3.5):
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, img, title in zip(axes.flat, images, titles):
        if img.ndim == 2:
            # cố định thang 0-255 để so sánh độ sáng giữa các ảnh xám
            ax.imshow(img, cmap='gray', vmin=0, vmax=255)
        else:
            ax.imshow(img)
        ax.set_title(title, fontsize=10)
    plt.tight_layout()
    if save:
        fig.savefig(OUT_DIR / save, dpi=100, bbox_inches='tight')
    plt.show()
    plt.close(fig)

## 1. Load ảnh màu

Dùng 5 ảnh:
- `tulips.jpg`, `parrot.jpg`, `peppers.jpg`: ảnh tự nhiên nhiều màu, lấy từ Wikimedia Commons (giấy phép CC0):
  [Tulip field](https://commons.wikimedia.org/wiki/File:Tulip_Field_Tulips_Red_Holland_Nature_Flowers.jpg), [Macaw parrot](https://commons.wikimedia.org/wiki/File:Macaw_parrot_sitting_on_a_tree_branch.jpg_.jpg), [Bell peppers](https://commons.wikimedia.org/wiki/File:Colorful_Bell_Peppers_(Unsplash).jpg).
- `sample.png`: logo BK, ảnh ít màu (chủ yếu xanh dương và trắng), có kênh alpha.
- `chart`: bảng màu tự tạo, gồm các màu cơ bản và một hàng các màu khác nhau nhưng có cùng độ sáng.

Ảnh màu được lưu dưới dạng mảng `(H, W, 3)` kiểu `uint8`, mỗi pixel là bộ ba `[R, G, B]` trong khoảng 0..255.

In [ ]:
def make_color_chart(cell=100):
    colors = [
        [(255, 0, 0), (0, 255, 0), (0, 0, 255)],        # đỏ, xanh lá, xanh dương
        [(255, 255, 0), (0, 255, 255), (255, 0, 255)],  # vàng, cyan, magenta
        [(255, 255, 255), (128, 128, 128), (0, 0, 0)],  # trắng, xám, đen
        [(255, 0, 0), (0, 130, 0), (50, 66, 195)],      # 3 màu khác nhau, cùng độ sáng ~76
    ]
    chart = np.zeros((len(colors) * cell, 3 * cell, 3), np.uint8)
    for i, row in enumerate(colors):
        for j, c in enumerate(row):
            chart[i * cell:(i + 1) * cell, j * cell:(j + 1) * cell] = c
    return chart


images = {
    'tulips': load_rgb(IN_DIR / 'tulips.jpg'),
    'parrot': load_rgb(IN_DIR / 'parrot.jpg'),
    'peppers': load_rgb(IN_DIR / 'peppers.jpg'),
    'logo': load_rgb(IN_DIR / 'sample.png'),
    'chart': make_color_chart(),
}

for name, img in images.items():
    print(f'{name:8s} shape={img.shape}  dtype={img.dtype}  pixel[0,0]={img[0, 0]}')

show(list(images.values()), list(images.keys()), cols=5, save='01_inputs.png')

Lưu ý: nếu hiển thị trực tiếp ảnh đọc bằng `cv2.imread` (thứ tự BGR) bằng matplotlib (hiểu là RGB) thì kênh R và B bị đổi chỗ, hoa tulip đỏ sẽ thành xanh dương.

In [ ]:
bgr = cv2.imread(str(IN_DIR / 'tulips.jpg'))
show([bgr, images['tulips']], ['Hiển thị sai (BGR)', 'Hiển thị đúng (RGB)'], cols=2, size=5)

## 2. Chuyển ảnh màu sang ảnh xám

Ảnh xám chỉ có 1 giá trị độ sáng cho mỗi pixel, nên cần đưa 3 giá trị (R, G, B) về 1 giá trị Y. Hai cách:

- Trung bình cộng: $Y = (R + G + B) / 3$
- Trung bình có trọng số (chuẩn ITU-R BT.601, OpenCV cũng dùng): $Y = 0.299R + 0.587G + 0.114B$

Trọng số của G lớn nhất vì mắt người nhạy nhất với ánh sáng xanh lá, kém nhạy nhất với xanh dương. Tổng các trọng số bằng 1 nên trắng vẫn là 255, đen vẫn là 0.

In [ ]:
def gray_average(rgb):
    return rgb.astype(np.float32).mean(axis=2)


def gray_luminance(rgb):
    w = np.array([0.299, 0.587, 0.114], dtype=np.float32)
    return rgb.astype(np.float32) @ w


# so sánh với hàm có sẵn của OpenCV
for name, img in images.items():
    ours = np.round(gray_luminance(img)).astype(np.uint8)
    ref = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    print(f'{name:8s} chênh lệch lớn nhất so với cv2: {np.abs(ours.astype(int) - ref).max()}')

for name in ['tulips', 'parrot', 'chart']:
    img = images[name]
    show([img, gray_average(img), gray_luminance(img)],
         [name, 'Trung bình (R+G+B)/3', 'Có trọng số (BT.601)'],
         cols=3, save=f'02_gray_{name}.png')

In [ ]:
colors = {'Đỏ': (255, 0, 0), 'Xanh lá': (0, 255, 0), 'Xanh dương': (0, 0, 255),
          'Xanh lá đậm': (0, 130, 0), 'Tím xanh': (50, 66, 195)}
print(f'{"Màu":12s} {"RGB":16s} {"Trung bình":>10s} {"BT.601":>8s}')
for name, c in colors.items():
    px = np.array([[c]], np.uint8)
    print(f'{name:12s} {str(c):16s} {gray_average(px)[0, 0]:10.1f} {gray_luminance(px)[0, 0]:8.1f}')

**Nhận xét:**
- Với công thức trung bình, đỏ, xanh lá và xanh dương thuần đều cho cùng mức xám 85, trong khi mắt thấy xanh lá sáng hơn xanh dương rất nhiều. Công thức BT.601 cho kết quả gần với cảm nhận hơn: xanh lá 150, đỏ 76, xanh dương 29.
- Ở ảnh tulips, hoa đỏ và bầu trời xanh đều khá tối trong ảnh xám vì R và B có trọng số nhỏ, vùng sáng nhất là đường đất màu nâu vàng. Hoa và trời có màu rất khác nhau nhưng mức xám lại gần nhau.
- Hàng cuối của bảng màu: ba màu đỏ, xanh lá đậm và tím xanh khác hẳn nhau nhưng đều cho mức xám khoảng 76, nên trong ảnh xám không phân biệt được. Chuyển sang xám là phép chiếu từ 3 chiều xuống 1 chiều, thông tin về màu bị mất.

## 3. Chuyển ảnh xám sang ảnh màu

Chuyển ngược chỉ là nhân bản kênh xám thành 3 kênh: R = G = B = Y. Mảng có dạng `(H, W, 3)` nhưng ảnh vẫn là xám, vì mọi điểm có R = G = B đều nằm trên đường chéo xám của khối màu RGB.

Không thể khôi phục màu gốc: một phương trình $Y = 0.299R + 0.587G + 0.114B$ có 3 ẩn nên có vô số nghiệm (R, G, B). Muốn có màu phải "đoán" (colorization bằng học máy) hoặc tô màu giả (pseudo-color) theo mức xám.

In [ ]:
img = images['tulips']
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
gray3 = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)      # tương đương np.stack([gray] * 3, axis=2)
pseudo = cv2.applyColorMap(gray, cv2.COLORMAP_JET)[:, :, ::-1]

print('gray  :', gray.shape, '->  gray3:', gray3.shape)
print('R == G == B:', np.array_equal(gray3[:, :, 0], gray3[:, :, 1]) and np.array_equal(gray3[:, :, 1], gray3[:, :, 2]))
print('Sai số trung bình so với ảnh gốc: %.1f / 255' % np.abs(img.astype(int) - gray3).mean())

show([img, gray3, pseudo], ['Ảnh gốc', 'Xám -> màu (R=G=B)', 'Tô màu giả (JET)'],
     cols=3, save='03_gray_to_color.png', size=4.5)

**Nhận xét:** ảnh sau khi chuyển màu → xám → màu vẫn là ảnh xám, sai số so với ảnh gốc khác 0 rõ rệt. Ảnh tô màu giả có màu nhưng màu chỉ thể hiện mức sáng (tối là xanh dương, sáng là vàng đỏ), không phải màu thật của vật: hoa đỏ bị tô thành xanh dương, đường đất thành vàng cam.

## 4. Tách các kênh màu R, G, B

Theo slide, ảnh màu là 3 hàm ghép lại: $f(x,y) = [r(x,y), g(x,y), b(x,y)]$. Mỗi kênh là một ma trận `(H, W)` giống ảnh xám, nên có thể hiển thị bằng thang xám: vùng càng sáng thì thành phần màu đó càng nhiều.

In [ ]:
for name in ['tulips', 'parrot', 'peppers', 'logo', 'chart']:
    img = images[name]
    R, G, B = cv2.split(img)          # tương đương img[:, :, 0], img[:, :, 1], img[:, :, 2]
    show([img, R, G, B], [name, 'Kênh R', 'Kênh G', 'Kênh B'], save=f'04_channels_{name}.png')

Để dễ hình dung, có thể hiển thị mỗi kênh bằng màu của nó (giữ một kênh, hai kênh còn lại bằng 0):

In [ ]:
def keep_channel(rgb, c):
    out = np.zeros_like(rgb)
    out[:, :, c] = rgb[:, :, c]
    return out


img = images['tulips']
show([img] + [keep_channel(img, c) for c in range(3)],
     ['tulips', 'Chỉ R', 'Chỉ G', 'Chỉ B'], save='04_channels_tinted.png')

**Nhận xét:**
- Tulips: hoa đỏ sáng ở kênh R, tối ở G và B. Bầu trời sáng ở kênh B, tối ở kênh R. Đường đất màu nâu vàng sáng ở R và G, tối hơn ở B.
- Parrot: lông xanh dương sáng ở kênh B, khá sáng ở G và tối ở R. Phần ngực vàng cam sáng nhất ở R, trung bình ở G, tối ở B (vàng = đỏ + xanh lá). Nền trắng sáng ở cả 3 kênh.
- Peppers: ớt đỏ chỉ sáng ở R, ớt vàng sáng ở R và G, còn kênh B gần như tối toàn bộ.
- Logo: gần như không có đỏ nên kênh R tối ở toàn bộ phần logo. Hai tông xanh (đậm và nhạt) khác nhau rõ nhất ở kênh G, kênh B đều khá sáng. Ảnh ít màu nên thông tin chủ yếu nằm ở 2 kênh G và B.
- Bảng màu: vùng trắng sáng ở cả 3 kênh, đen tối ở cả 3, xám 128 có cùng giá trị 128 ở cả 3 kênh.

## 5. Kết hợp các kênh màu

### 5.1 Tái tạo ảnh ban đầu

In [ ]:
for name, img in images.items():
    R, G, B = cv2.split(img)
    rec = cv2.merge([R, G, B])
    print(f'{name:8s} giống hệt ảnh gốc: {np.array_equal(rec, img)}')

Ghép lại 3 kênh đúng thứ tự cho ảnh giống hệt ảnh gốc (sai số bằng 0). Tách kênh không làm mất thông tin, khác với việc chuyển sang ảnh xám.

### 5.2 Hoán đổi kênh

Có 3! = 6 cách sắp xếp 3 kênh, trong đó 1 cách là ảnh gốc.

In [ ]:
orders = {'RGB': [0, 1, 2], 'RBG': [0, 2, 1], 'GRB': [1, 0, 2],
          'GBR': [1, 2, 0], 'BRG': [2, 0, 1], 'BGR': [2, 1, 0]}

for name in ['tulips', 'parrot', 'chart']:
    img = images[name]
    show([img[:, :, o] for o in orders.values()], list(orders.keys()),
         cols=6, size=2.8, save=f'05_swap_{name}.png')

**Nhận xét:**
- Kết quả dự đoán được bằng phép cộng màu. Ví dụ với BGR (đổi R và B): hoa đỏ thành xanh dương, bầu trời xanh thành cam nâu, vẹt xanh dương thành vàng cam còn phần ngực vàng cam thành xanh dương.
- Vùng trắng, đen, xám (R = G = B) không thay đổi ở mọi cách hoán đổi. Có thể thấy rõ ở hàng thứ ba của bảng màu và nền trắng của ảnh parrot.
- Hoán đổi kênh không làm mất thông tin: đổi lại đúng thứ tự thì được ảnh gốc.

### 5.3 Thiếu một kênh màu

In [ ]:
def drop_channel(rgb, c):
    out = rgb.copy()
    out[:, :, c] = 0
    return out


for name in ['tulips', 'parrot', 'chart']:
    img = images[name]
    show([img] + [drop_channel(img, c) for c in range(3)],
         [name, 'Không có R', 'Không có G', 'Không có B'], save=f'05_drop_{name}.png')

# độ sáng trung bình (mức xám) sau khi bỏ từng kênh
print(f'{"":8s} {"gốc":>6s} {"bỏ R":>6s} {"bỏ G":>6s} {"bỏ B":>6s}')
for name in ['tulips', 'parrot', 'peppers']:
    img = images[name]
    vals = [img] + [drop_channel(img, c) for c in range(3)]
    print(f'{name:8s}' + ''.join(f'{cv2.cvtColor(v, cv2.COLOR_RGB2GRAY).mean():7.1f}' for v in vals))

**Nhận xét:**
- Bỏ R: ảnh ngả màu cyan (G + B), hoa đỏ gần như thành đen.
- Bỏ G: ảnh ngả màu magenta (R + B), lá cây gần như biến mất. Theo bảng số liệu, bỏ G làm độ sáng giảm nhiều nhất ở cả 3 ảnh vì G có trọng số lớn nhất (0.587). Riêng tulips, bỏ R cũng tối gần bằng vì ảnh chủ yếu là màu đỏ.
- Bỏ B: ảnh ngả màu vàng (R + G), bầu trời chuyển thành xanh rêu, vùng trắng thành vàng. Độ sáng giảm ít nhất vì B có trọng số nhỏ (0.114).
- Vật mang màu của kênh bị bỏ sẽ tối đi hoặc đổi màu, còn vật không chứa kênh đó thì gần như không đổi (ví dụ hoa đỏ khi bỏ B).

### 5.4 Thay thế và kết hợp kênh từ nhiều nguồn

In [ ]:
img = images['tulips']
R, G, B = cv2.split(img)
h, w = R.shape
full = np.full_like(R, 255)

# kênh lấy từ ảnh khác (đưa về cùng kích thước)
g_parrot = cv2.cvtColor(cv2.resize(images['parrot'], (w, h)), cv2.COLOR_RGB2GRAY)
g_peppers = cv2.cvtColor(cv2.resize(images['peppers'], (w, h)), cv2.COLOR_RGB2GRAY)
g_tulips = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

results = {
    'R = 255': cv2.merge([full, G, B]),
    'R = G': cv2.merge([G, G, B]),
    'B = 255 - B (âm bản kênh B)': cv2.merge([R, G, 255 - B]),
    'Xám của 3 ảnh -> R, G, B': cv2.merge([g_tulips, g_parrot, g_peppers]),
}
show([img] + list(results.values()), ['tulips'] + list(results.keys()),
     cols=5, size=3.2, save='05_replace.png')

**Nhận xét:**
- `R = 255`: toàn ảnh ám đỏ, vùng tối thành đỏ, bầu trời chuyển sang hồng.
- `R = G`: hoa đỏ thành gần đen vì kênh G ở vùng hoa rất thấp, bầu trời chuyển sang tím nhạt.
- Âm bản kênh B: vùng trời (B lớn) mất xanh dương nên thành xanh rêu, vùng hoa (B nhỏ) nhận thêm nhiều xanh dương nên chuyển sang hồng tím.
- Ghép ảnh xám của 3 ảnh khác nhau vào 3 kênh: tạo ra ảnh chồng ba hình, mỗi hình hiện lên với một màu riêng. Đây cũng là cách ảnh màu Prokudin-Gorskii được tạo ra từ 3 tấm kính đen trắng (slide Chapter 1).

## 6. Mở rộng

### 6.1 Biến đổi màu bằng ma trận 3x3

Các phép ở trên đều có dạng $[R', G', B']^T = M \cdot [R, G, B]^T$ áp dụng cho từng pixel (point processing):
- hoán đổi kênh: $M$ là ma trận hoán vị,
- bỏ một kênh: $M$ là ma trận đường chéo có một phần tử bằng 0,
- chuyển sang xám: cả 3 hàng của $M$ bằng $[0.299, 0.587, 0.114]$.

Từ đó có thể tạo thêm hiệu ứng khác, ví dụ ảnh màu nâu cổ điển (sepia).

In [ ]:
def apply_matrix(rgb, M):
    out = rgb.astype(np.float32) @ np.asarray(M, np.float32).T
    return np.clip(out, 0, 255).astype(np.uint8)   # tránh tràn số uint8


matrices = {
    'Hoán đổi BGR': [[0, 0, 1], [0, 1, 0], [1, 0, 0]],
    'Bỏ kênh B': [[1, 0, 0], [0, 1, 0], [0, 0, 0]],
    'Xám': [[0.299, 0.587, 0.114]] * 3,
    'Sepia': [[0.393, 0.769, 0.189], [0.349, 0.686, 0.168], [0.272, 0.534, 0.131]],
}
img = images['parrot']
show([img] + [apply_matrix(img, M) for M in matrices.values()], ['parrot'] + list(matrices.keys()),
     cols=5, size=3.2, save='06_matrix.png')

### 6.2 Không gian màu YCrCb

Trong RGB, độ sáng và màu sắc nằm lẫn trong cả 3 kênh. Không gian YCrCb tách riêng độ sáng (Y) và thông tin màu (Cr, Cb). Kênh Y dùng đúng công thức BT.601, nên chính là ảnh xám ở mục 2.

In [ ]:
img = images['tulips']
Y, Cr, Cb = cv2.split(cv2.cvtColor(img, cv2.COLOR_RGB2YCrCb))
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
print('Chênh lệch lớn nhất giữa kênh Y và ảnh xám:', np.abs(Y.astype(int) - gray).max())   # chỉ do làm tròn

# giữ Y, đặt Cr = Cb = 128 (không có màu) -> ảnh xám
no_color = cv2.cvtColor(cv2.merge([Y, np.full_like(Cr, 128), np.full_like(Cb, 128)]), cv2.COLOR_YCrCb2RGB)
show([img, Y, Cr, Cb, no_color], ['tulips', 'Y', 'Cr', 'Cb', 'Chỉ giữ Y'], cols=5, size=3.2, save='06_ycrcb.png')

**Nhận xét:** kênh Y trùng với ảnh xám (chỉ lệch tối đa 1 đơn vị do làm tròn). Kênh Cr sáng ở vùng hoa đỏ, kênh Cb sáng hơn ở vùng trời xanh. Bỏ Cr và Cb thì còn lại đúng ảnh xám, tức là chuyển sang ảnh xám tương đương với giữ lại độ sáng và bỏ đi phần màu.

## 7. Nhận xét chung

- Ảnh xám là ma trận `(H, W)`, ảnh màu là 3 ma trận `(H, W)` ghép lại. Mỗi kênh màu bản chất là một ảnh xám.
- Chuyển màu → xám làm mất thông tin: nhiều màu khác nhau có cùng mức xám. Công thức có trọng số (BT.601) cho độ sáng gần với cảm nhận của mắt hơn công thức trung bình.
- Chuyển xám → màu chỉ đổi định dạng (R = G = B), không khôi phục được màu gốc.
- Tách và ghép kênh không mất thông tin. Hoán đổi, bỏ hoặc thay thế kênh làm màu thay đổi theo đúng quy luật cộng màu RGB. Vùng trung tính (trắng, xám, đen) không đổi khi hoán đổi kênh.
- Ảnh ít màu (logo) cho thấy ít khác biệt giữa các kênh, ảnh nhiều màu (tulips, parrot) thể hiện rõ vai trò của từng kênh.